# 2.7 Facet spaces and hybrid methods

Mixed methods for second order problems lead to saddle point problems, and indefinite matrices. By hybridization one obtains a positive definite system again. Its structure is similar to the non-conforming $P^1$ method, but hybridization works for any order. See the textbook by Brezzi and Fortin.

One gives up the normal continuity of the $H(div)$ variable, and enforces it again by a Lagrange multiplier. This leads to the following discrete system:

Find $(\sigma, u, \widehat u) \in \Sigma_h \times V_h \times F_h$ such that

$$
\DeclareMathOperator{\Div}{div}
\begin{array}{ccccccll}
\int \lambda^{-1} \sigma \tau & + & \sum_T \int_T \Div \tau \, u & - & \sum_F \int_F [\tau_n] \widehat u & = & 0 & \forall \, \tau \in \Sigma_h \\
\sum_T \int_T \Div \sigma \, v &&&&& = & -\int f v & \forall \, v \in V_h \\
-\sum_F \int_F [ \sigma_n ] \, \widehat v &&&&& = & -\int_{\Gamma_N} g \widehat v & \forall \, \widehat v \in F_h
\end{array}
$$

where $\Sigma_h$ is a discontinuous $H(div)$ finite element space, $V_h$ a sub-space of $L_2$, and $F_h$ consists of polynomials on every edge. The Lagrange multiplier $\widehat u$ is an approximation to $u$ on the edges. Thus, the Dirichlet condition $\widehat u = u_D$ on $\Gamma_D$ is an essential boundary condition for the facet space.

In [ ]:
from ngsolve import *
from ngsolve.webgui import Draw
mesh = Mesh(unit_square.GenerateMesh(maxh=0.2))

The same example as in [unit 2.5](../unit-2.5-mixed/mixed.ipynb):

In [ ]:
source = sin(pi*x)
ud = CF(5)
g = mesh.BoundaryCF( {"left" : y*(1-y)}, default=0)
lam = 10

Define the spaces:

* The `Discontinuous` wrapper generates an element-wise discontinuous version of a space
* `FacetFESpace` lives only on facets (i.e. faces in 3D, edges in 2D, points in 1D)

In [ ]:
order = 3
V = Discontinuous (HDiv(mesh, order=order))
Q = L2(mesh, order=order-1)
F = FacetFESpace(mesh, order=order)
X = V*Q*F
print ("sigmadofs:", X.Range(0))
print ("udofs:    ", X.Range(1))
print ("uhatdofs: ", X.Range(2))

Define the variational equation. The jump term is rewritten as

$$
\sum_F \int_F [\sigma_n] \widehat v = \sum_T \int_{\partial T} \sigma_n \widehat v
$$

Integrals over the element boundaries are obtained by `dx(element_boundary=True)`:

In [ ]:
(sigma,u,uhat), (tau,v,vhat) = X.TnT()
n = specialcf.normal(mesh.dim)

lhs = (1/lam * sigma*tau + div(sigma)*v + div(tau)*u) * dx \
    + (-sigma*n*vhat-tau*n*uhat) * dx(element_boundary=True)
rhs = -source*v * dx - g*vhat * ds

Now all unknowns of $\sigma$ and $u$ are element-local, and `Solve` eliminates them by static condensation. The remaining system for the facet variable $\widehat u$ is symmetric and positive definite. We solve it by a conjugate gradient iteration with a BDDC preconditioner. The boundary condition is now posed for the facet variable:

In [ ]:
gf = Solve(lhs==rhs, uhat[BND("bottom")] == ud,
           preconditioners.BDDC(),
           solvers.CGSolver.Creator(printrates=True))

In [ ]:
Draw (gf[sigma], mesh, "sigma")
Draw (gf[u], mesh, "u");

With `condense=False` the local unknowns are kept, and the sparse direct solver has to deal with the big saddle point system:

In [ ]:
gf2 = Solve(lhs==rhs, uhat[BND("bottom")] == ud, condense=False)
print ("difference:", sqrt(Integrate((gf[u]-gf2[u])**2, mesh)))

## Behind the scenes

`Solve` has set up a `BilinearForm` with the flag `condense`, and has performed the following steps, see also [unit 1.4](../unit-1.4-staticcond/staticcond.ipynb). Now the Dirichlet boundary is a property of the facet space:

In [ ]:
F = FacetFESpace(mesh, order=order, dirichlet="bottom")
X = V*Q*F
(sigma,u,uhat), (tau,v,vhat) = X.TnT()

a = BilinearForm(X, condense=True)
a += (1/lam * sigma*tau + div(sigma)*v + div(tau)*u) * dx
a += (-sigma*n*vhat-tau*n*uhat) * dx(element_boundary=True)
c = preconditioners.BDDC(a)

f = LinearForm(-source*v * dx - g*vhat * ds)

a.Assemble()
f.Assemble()
print ("A non-zero", a.mat.nze)

The matrix `a.mat` is the Schur complement with respect to the facet variable. We modify the right hand side accordingly, solve for the facet variable, and finally recover the local unknowns:

In [ ]:
gfu = GridFunction(X)
gfu.components[2].Set(ud, BND)

fmod = (f.vec + a.harmonic_extension_trans * f.vec).Evaluate()
solvers.CG(mat=a.mat, pre=c, rhs=fmod, sol=gfu.vec, initialize=False)

gfu.vec.data += a.harmonic_extension * gfu.vec
gfu.vec.data += a.inner_solve * f.vec

print ("difference:", sqrt(Integrate((gfu.components[1]-gf.components[1])**2, mesh)))